In [2]:
from scipy.io import loadmat
import os

path = r"C:\S.R\SCIBS\data"
file_path = os.path.join(path, "testhead.mat")
data = loadmat(file_path)

In [3]:
dataB = os.path.join(path, "testhead_E.mat")
dataB = loadmat(dataB)

In [4]:
import meshio
import numpy as np
from scipy.io import savemat

# read the mesh
mesh = meshio.read("ernie.msh")

In [5]:
nodes = mesh.points.astype(np.float64)

# Collect all tetrahedral elements
tetra = []
for block in mesh.cells:
    if block.type == "tetra":
        tetra.append(block.data)


cells = np.vstack(tetra).astype(np.int32) + 1  # MATLAB uses 1-based indexing

Geometry = {
    "node": nodes,
    "cell": cells
}

savemat("ernie.mat", {"Geometry": Geometry}, do_compression=True)

In [15]:

matZero = loadmat("ernie.mat")

In [19]:
Geometry = matZero["Geometry"]

nodes = Geometry["node"][0, 0]
cells = Geometry["cell"][0, 0]

print(nodes.shape)
print(cells.shape)

(797287, 3)
(4463007, 4)


In [16]:
matZero

{'__header__': b'MATLAB 5.0 MAT-file Platform: nt, Created on: Sat Aug  8 09:09:21 2026',
 '__version__': '1.0',
 '__globals__': [],
 'Geometry': array([[(array([[-20.52151496,  20.47415707, -34.18745585],
                [ -2.29272866,  24.35138563, -29.00224944],
                [  5.07152245,  15.16820301, -24.27001108],
                ...,
                [ 12.10228707, -13.81908693,  68.38677082],
                [-28.6109549 ,  21.70048414, -12.09300368],
                [  4.77921221,  -1.18387887, -75.82354108]], shape=(797287, 3)), array([[215291, 572228, 283894, 215269],
                [ 47960,   9527, 678852,  47957],
                [751495, 772399, 483550, 445221],
                ...,
                [165133, 165154, 753680, 165225],
                [106393, 498181, 796650, 551904],
                [377755, 524772,  64020, 377834]], shape=(4463007, 4), dtype=int32))                                     ]],
       dtype=[('node', 'O'), ('cell', 'O')])}

In [17]:
matOne = loadmat("TH.mat")

In [18]:
matOne

{'__header__': b'MATLAB 5.0 MAT-file Platform: nt, Created on: Wed Jul 22 09:25:50 2026',
 '__version__': '1.0',
 '__globals__': [],
 'Geometry': array([[(array([[  0.47633432,  -1.61650072,  87.98533333],
                [ -2.04157568,   1.33094858,  87.956     ],
                [  2.82164918,   1.08502642,  87.92666667],
                ...,
                [  1.21090485,  -3.44764886, -11.80314667],
                [ -1.53883706,   0.85042595, -11.93651556],
                [  0.        ,   0.        ,   0.        ]], shape=(11181, 3)), array([[10923, 10256, 10222, 10277],
                [ 8764, 10256, 10222, 10277],
                [ 7947,  3921,  4154,  4010],
                ...,
                [ 2085,  2229,  7064,  7098],
                [ 2085,  2174,  2229,  7098],
                [ 2085,  7064,  7098,  7009]], shape=(52448, 4), dtype=int32))                                      ]],
       dtype=[('node', 'O'), ('cell', 'O')])}

In [20]:
import numpy as np

nodes = matZero["Geometry"]["node"][0, 0]
cells = matZero["Geometry"]["cell"][0, 0]

# Select 52,448 tetrahedra
rng = np.random.default_rng(42)

selected = rng.choice(
    len(cells),
    size=52448,
    replace=False
)

cells_small = cells[selected]

# Find nodes actually used by those cells
used_nodes = np.unique(cells_small)

# MATLAB -> Python indexing
used_nodes_python = used_nodes - 1

# Extract those nodes
nodes_small = nodes[used_nodes_python]

# Remap cell indices to the new node numbering
node_map = {
    old: new
    for new, old in enumerate(used_nodes)
}

cells_small = np.vectorize(node_map.get)(cells_small)

In [26]:
import numpy as np
import scipy.io as sio

# 1. Extract nodes and compute bounding boxes
nodes_zero = matZero['Geometry'][0, 0]['node']
nodes_one = matOne['Geometry'][0, 0]['node']

min_zero, max_zero = nodes_zero.min(axis=0), nodes_zero.max(axis=0)
min_one, max_one = nodes_one.min(axis=0), nodes_one.max(axis=0)

size_zero = max_zero - min_zero
size_one = max_one - min_one
size_zero_safe = np.where(size_zero == 0, 1.0, size_zero)

# 2. Rescale and shift nodes_zero to match matone extent
scale_factors = size_one / size_zero_safe
nodes_resized = (nodes_zero - min_zero) * scale_factors + min_one

# 3. Update node array inside the original dictionary structure
matZero['Geometry'][0, 0]['node'] = nodes_resized

# 4. Save to a new .mat file
sio.savemat('ernie.mat', matZero)

In [28]:
import numpy as np
import scipy.io as sio
from scipy.spatial import cKDTree

# 1. Access the Geometry array using 'matZero'
geom = matZero['Geometry'][0, 0]
nodes = geom['node'].copy()  # shape: (797287, 3)

# 2. Center the mesh along the X-axis (mid-sagittal plane at X = 0)
x_mid = (nodes[:, 0].min() + nodes[:, 0].max()) / 2.0
nodes[:, 0] -= x_mid

# 3. Create a flipped copy along X to locate mirror node pairs
mirrored_nodes = nodes.copy()
mirrored_nodes[:, 0] *= -1

# 4. Map each node to its closest opposite-side counterpart
tree = cKDTree(mirrored_nodes)
_, indices = tree.query(nodes)

# 5. Average positions to remove lopsided/asymmetric features
nodes_sym = (nodes + mirrored_nodes[indices]) / 2.0

# Enforce correct sign on X coordinates for left/right sides
nodes_sym[:, 0] = np.where(nodes[:, 0] >= 0, np.abs(nodes_sym[:, 0]), -np.abs(nodes_sym[:, 0]))

# 6. Update node coordinates directly inside matZero
matZero['Geometry'][0, 0]['node'] = nodes_sym

# 7. Export back to MATLAB .mat file
sio.savemat('matZero_symmetric.mat', matZero)

print("Successfully generated 'matZero_symmetric.mat'")

Successfully generated 'matZero_symmetric.mat'


In [29]:
import numpy as np
import scipy.io as sio

# 1. Load the MATLAB file
mat_path = r"C:\S.R\SCIBS\data\testhead.mat"
data = sio.loadmat(mat_path)

# Extract nodes and cells (handling case variations in key names)
geom_key = 'Geometry' if 'Geometry' in data else 'geometry'
geom = data[geom_key][0, 0]

nodes = geom['node']  # shape: (N, 3)
cells = geom['cell']  # shape: (M, 4)

# 2. Adjust for 1-based indexing if necessary
if cells.min() >= 1:
    cells_zero = cells - 1
else:
    cells_zero = cells

# 3. Write out .pts and .tet files
base_path = r"C:\S.R\SCIBS\data\testhead"

# Save nodes to .pts
np.savetxt(f"{base_path}.pts", nodes, fmt="%.8f")

# Save tetrahedra connectivity to .tet
np.savetxt(f"{base_path}.tet", cells_zero, fmt="%d")

print(f"Successfully converted {mat_path} to {base_path}.pts and {base_path}.tet")

Successfully converted C:\S.R\SCIBS\data\testhead.mat to C:\S.R\SCIBS\data\testhead.pts and C:\S.R\SCIBS\data\testhead.tet
